# Document Store Query Results

Exercises what the **github** and **research_library** document stores actually return.
These are the two stores that stay on Chroma — FRED and ETF time series move to Postgres.

The point of this notebook is to make visible the things that were invisible while document
search was failing:

- what a query string turns into (`where` filter + cleaned query)
- whether a filter matched nothing, and whether the unfiltered retry saved it
- the metadata header now prepended to each chunk
- the relevance score the ranker assigns to each chunk
- duplicate chunks of the same file inside one result set

## Setup

**Load `.env`, not `.keys`.** `set_*_env(".keys")` loads API keys but no model
configuration, so `LLM_MODEL_PROVIDER` and `SCORING_LLM_MODEL` fall back to the factory
defaults and you benchmark a different model than the app runs. This cost a day of wrong
measurements on 2026-10-17.

In [ ]:
%reload_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path
from collections import Counter

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from dotenv import load_dotenv
load_dotenv(Path("../../.env").resolve())

DB_PATH = str(Path("../../.db").resolve())

from langchain_core.messages import HumanMessage, SystemMessage
from apps.agentic.agents.document.code_repo_agent import CodeRepoAgent
from apps.agentic.agents.document.research_library_agent import ResearchLibraryAgent
from apps.agentic.core.agents.query_filters import build_filter_and_query
from apps.agentic.core.llm_factory import scoring_llm_model, agent_llm_model

SEP = "\n\n-----\n\n"

print("provider :", os.getenv("LLM_MODEL_PROVIDER"))
print("agent    :", os.getenv("AGENT_LLM_MODEL"))
print("ranker   :", os.getenv("SCORING_LLM_MODEL"), "temp", scoring_llm_model().temperature)

## Store sizes

In [ ]:
from apps.agentic.core.document_loaders.github_document_loader import GitHubChromaDocumentLoader
from apps.agentic.core.document_loaders.research_library_document_loader import ResearchLibraryChromaDocumentLoader

STORES = {
    "github":           GitHubChromaDocumentLoader(DB_PATH),
    "research_library": ResearchLibraryChromaDocumentLoader(DB_PATH),
}
for name, loader in STORES.items():
    print(f"{name:<18} {loader.vectorstore._collection.count():>8,} chunks")

## Metadata keys actually present

Only keys at 100% are safe to use in a filter or a document header — a key missing from one
document is a silent zero-result or a format error. `collection.get()` binds one SQL
variable per returned row and SQLite caps at 32766, so pull in pages.

In [ ]:
def all_metadatas(collection, batch=20_000):
    out, offset, total = [], 0, collection.count()
    while offset < total:
        got = collection.get(limit=batch, offset=offset, include=["metadatas"]).get("metadatas") or []
        if not got:
            break
        out.extend(m for m in got if m)
        offset += len(got)
    return out

for name, loader in STORES.items():
    metas = all_metadatas(loader.vectorstore._collection)
    n = len(metas)
    keys = Counter(k for m in metas for k in m)
    print(f"\n{name}  ({n:,} chunks)")
    for k, c in sorted(keys.items()):
        flag = "" if c == n else "   <-- PARTIAL"
        print(f"    {k:<22} {100*c/n:6.2f}%{flag}")

## 1. What does a query string become?

Inline qualifiers are parsed out into a Chroma `where` filter; the rest is the text that
gets embedded. Every qualifier is an **exact, case-sensitive equality** match — a near-miss
returns zero documents, not fewer.

In [ ]:
PROBES = [
    "repo:zgomot Where is MIDI output handled?",
    "account:troystribling repo:zgomot ext:.rb after:2014-01-01 note_on handler",
    "repo:yada where is the orchestrator defined",              # yada indexes to 0 chunks
    'authors:"William H. Matthaeus" magnetic helicity decay',    # single name, never matches
    'authors:"Troy Stribling, William H. Matthaeus" magnetic helicity decay',
    "shelf:publications statistical mechanics applied to MHD",
    "title:Thermodynamics definition of a Carnot Cycle",
    "topic:brownian how is fBm simulated",                       # topic is stripped, not filtered
]
for p in PROBES:
    clean, where = build_filter_and_query(p)
    print(f"{p}\n    where -> {where}\n    query -> {clean!r}\n")

## 2. Raw retrieval

What MMR returns before any ranking. Watch for the same `path` appearing more than once —
duplicate chunks of one file consume result slots.

In [ ]:
def retrieved(agent, question, show=8):
    hits = agent.retriever.invoke(question)
    key = "file_path" if "file_path" in (hits[0].metadata if hits else {}) else "path"
    paths = [h.metadata.get(key) for h in hits]
    dupes = {p: c for p, c in Counter(paths).items() if c > 1}
    print(f"{len(hits)} hits" + (f"   DUPLICATES: {dupes}" if dupes else ""))
    for i, h in enumerate(hits[:show], 1):
        print(f"  {i}. {h.metadata.get(key)}")
        print(f"     {h.page_content.strip()[:88]!r}")
    return hits

gh = CodeRepoAgent(query={"repo": "zgomot"})
rl = ResearchLibraryAgent(query={})

print("=== github ===");           _ = retrieved(gh, "Which file handles MIDI output?")
print("\n=== research_library ==="); _ = retrieved(rl, "What algorithms are used for generating fractional Brownian motion?")

## 3. The context the grader and model actually see

Each chunk is prepended with a metadata header (`DOCUMENT_HEADER_KEYS`). Without it the
model is told to cite file paths it cannot see, and the ranker judges chunks with no idea
where they came from.

In [ ]:
def context_for(agent, question):
    return agent._retrieve({"messages": [HumanMessage(content=question)]})["messages"][-1].content

ctx = context_for(gh, "Which file handles MIDI output?")
print(ctx[:600])

## 4. Ranked results with scores

One LLM call scores every candidate against the others. Selection is then
`grade_min_score` (absolute — is any of this on-topic at all) then `grade_relative_floor`
(relative to the best score — how many are worth keeping) then `grade_top_n`.

The log line shows the full score vector, including chunks that were cut.

In [ ]:
def ranked(agent, question):
    ctx = context_for(agent, question)
    out = agent._grade_documents({"messages": [HumanMessage(content=question),
                                               SystemMessage(content=ctx)]})
    body = out["messages"][-1].content
    if body.startswith("(no "):
        print(f"  EMPTY: {body}")
        return []
    kept = [c for c in body.split(SEP) if c.strip()]
    for k in kept:
        header = k.split("\n")[1] if "\n" in k else k[:70]
        score  = k.rsplit("[relevance: ", 1)[-1].rstrip("]\n") if "[relevance:" in k else "?"
        print(f"  {score}  {header[:96]}")
    return kept

print("=== github ===");             _ = ranked(gh, "Which file handles MIDI output?")
print("\n=== research_library ==="); _ = ranked(rl, "What algorithms are used for generating fractional Brownian motion?")

## 5. Off-topic queries should come back empty

A ranker that always returns its top n would hand the model irrelevant context and invite
confabulation. `grade_min_score` is what prevents that.

In [ ]:
for q in ["How is OAuth token refresh implemented?",
          "Where is the Kubernetes deployment manifest?"]:
    print(f"{q}"); _ = ranked(gh, q); print()

## 6. Does a filter help or hurt?

An unsatisfiable filter used to return a confident empty answer. It now retries unfiltered
and logs `Filter matched nothing`. Compare the two paths directly.

In [ ]:
def compare_filter(agent_cls, question, where):
    for label, q in (("unfiltered", None), (f"filtered {where}", where)):
        a = agent_cls(query=q)
        hits = a.retriever.invoke(question)
        key = "file_path" if hits and "file_path" in hits[0].metadata else "path"
        top = hits[0].metadata.get(key) if hits else None
        print(f"  {label:<52} {len(hits)} hits   top={top}")

compare_filter(ResearchLibraryAgent, "magnetic helicity decay in MHD turbulence",
               {"authors": "William H. Matthaeus"})
print()
compare_filter(CodeRepoAgent, "where is the orchestrator defined", {"repo": "yada"})

## 7. End to end

The full graph: `retrieve -> grade -> generate`. Check the answer cites the metadata it was
asked to cite.

In [ ]:
def answer(agent, question, thread="nb-1"):
    out = agent._create_agent().invoke({"messages": [HumanMessage(content=question)]},
                                       config={"configurable": {"thread_id": thread}})
    return out["messages"][-1].content

print(answer(gh, "Which file handles MIDI output? Give the file path.", "nb-gh")[:800])

## 8. Batch harness

Run a list of queries and see, per query, how many were retrieved, how many survived
ranking, and whether the result was empty. Use this when changing `grade_relative_floor`
or the ranker prompt — a single query proves nothing.

In [ ]:
def sweep(agent, questions):
    rows = []
    for q in questions:
        ctx = context_for(agent, q)
        n_ret = len([c for c in ctx.split(SEP) if c.strip()])
        body = agent._grade_documents({"messages": [HumanMessage(content=q),
                                                    SystemMessage(content=ctx)]})["messages"][-1].content
        empty = body.startswith("(no ")
        n_kept = 0 if empty else len([c for c in body.split(SEP) if c.strip()])
        rows.append((q, n_ret, n_kept, empty))
    print(f"{'query':<58} {'ret':>4} {'kept':>5}  empty")
    for q, r, k, e in rows:
        print(f"{q[:56]:<58} {r:>4} {k:>5}  {'YES' if e else ''}")
    return rows

_ = sweep(gh, [
    "Which file handles MIDI output?",
    "Where is the CoreMIDI driver implemented?",
    "How are notes converted to MIDI pitch values?",
    "How is OAuth token refresh implemented?",
])